In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive', force_remount=True)
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'test-download-capitanata'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

# 02 - Download e Riepilogo dei dati satellitari per l'area di studio
Download delle 6 bande Sentinel-2 per l'intera Bounding Box della Capitanata per ciascun mese dell'anno.

#### Download dei dati satellitari (Bounding Box Capitanata)

In [ ]:
from pathlib import Path
from src.test_api2 import download_area

# Nuova cartella di output dedicata (preserva intatta 'sentinel2_data')
out_directory = DATA_DIR / "processed" / "sentinel2_capitanata_area"

# Coordinate esatte della sola Capitanata (EPSG:4326)
CAPITANATA_CONFIG = {
    "region": "italy_capitanata",
    "min_lon": 15.046692771569205,
    "min_lat": 41.08392693013771,
    "max_lon": 16.210767777473393,
    "max_lat": 41.92848568568325,
    "crs": "EPSG:4326",
}

capitanata_bbox = [
    CAPITANATA_CONFIG["min_lon"],
    CAPITANATA_CONFIG["min_lat"],
    CAPITANATA_CONFIG["max_lon"],
    CAPITANATA_CONFIG["max_lat"],
]
print(f"🗺️ BBox esatta Capitanata: {capitanata_bbox}")

# Avvio del download per la sola Capitanata per l'anno 2023
downloaded_files = download_area(
    name="capitanata",
    bbox=capitanata_bbox,
    year="2023",
    months=range(1, 13),
    out_dir=str(out_directory)
)

#### Riepilogo e anteprima visiva RGB

In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

out_directory = DATA_DIR / "processed" / "sentinel2_capitanata_area"
tif_files = sorted(list(out_directory.glob("*.tif")))

print("=" * 50)
print("📊 REPORT DATI SATELLITARI AREA CAPITANATA")
print("=" * 50)
print(f"📁 Cartella: {out_directory.resolve()}")
print(f"📦 File GeoTIFF mensili trovati: {len(tif_files)}")
for f in tif_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  • {f.name} ({size_mb:.1f} MB)")
print("=" * 50)

# Preview RGB di un mese estivo (es. mese 07 o il primo disponibile)
preview_candidates = [f for f in tif_files if "07" in f.name or "06" in f.name]
preview_tif = preview_candidates[0] if preview_candidates else (tif_files[0] if tif_files else None)

if preview_tif and preview_tif.exists():
    with rasterio.open(preview_tif) as src:
        print(f"\nVisualizzazione anteprima: {preview_tif.name}")
        print(f"  • Dimensioni: {src.width} x {src.height} pixel")
        print(f"  • Bande: {src.count} (B02, B03, B04, B08, B11, B12)")
        print(f"  • Proiezione: {src.crs}")

        # Lettura B04 (Rosso = banda 3), B03 (Verde = banda 2), B02 (Blu = banda 1)
        red = src.read(3)
        green = src.read(2)
        blue = src.read(1)

        rgb = np.dstack([red, green, blue])
        valid = rgb > 0
        if np.any(valid):
            p2, p98 = np.percentile(rgb[valid], (2, 98))
            rgb_scaled = np.clip((rgb - p2) / (p98 - p2 + 1e-5), 0, 1)

            plt.figure(figsize=(10, 10))
            plt.imshow(rgb_scaled)
            plt.title(f"Sentinel-2 RGB Naturale - Capitanata ({preview_tif.name})")
            plt.axis("off")
            plt.show()
else:
    print("Nessun file GeoTIFF disponibile per la preview grafica.")